# FastPower (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `FastPower`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/FastPower_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Fast exponentiation: FastPower

**Input:** a number $x$ and a natural number $n$. **Output:** $x^n$, by repeated squaring. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor + 2$ calls, at most $2(\lfloor\log_2 n\rfloor + 1)$ multiplications.

Pseudo-code: section 14.7 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile fast_power.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

int FloorLog2(int n)                               // floor(log2 n) with a loop (only for the formula of the steps)
{
    int k = 0;
    while (n > 1)
    {
        n = n / 2;
        k = k + 1;
    }
    return k;
}

// FastPower(x, n)   values that are too large overflow in C++ and Java, not in Python
long long FastPower(long long x, int n)
{
    Push(n); cnt[2]++;
    if (n == 0)
    {
        cnt[3]++;
        if (trace) Say(Str("FastPower(%lld, 0)", x), "base case, returns 1");
        Pop();
        return 1;
    }
    if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("%d != 0 => calls FastPower(%lld, %d)   (%d/2 = %d)", n, x, n / 2, n, n / 2));
    long long y = FastPower(x, n / 2); cnt[5]++;   // one recursive call only: the result is in y
    cnt[6]++;
    long long r;
    if (n % 2 == 0)
    {
        cnt[7]++;
        mults += 1;
        r = y * y;
        if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("y = %lld, %d is even => returns %lld * %lld = %lld", y, n, y, y, r));
    }
    else
    {
        cnt[9]++;
        mults += 2;
        r = x * y * y;
        if (trace) Say(Str("FastPower(%lld, %d)", x, n), Str("y = %lld, %d is odd => returns %lld * %lld * %lld = %lld", y, n, x, y, y, r));
    }
    Pop();
    return r;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: FastPower(3, 13)
    long long x = 3;
    int n = 13;
    Reset();
    trace = true;
    printf("%-28s| %s\n", "call", "action");
    long long r = FastPower(x, n);
    trace = false;
    printf("%lld^%d = %lld, calls = %lld, multiplications = %lld\n", x, n, r, calls, mults);
    PrintSteps({2, 3, 5, 6, 7, 9});
    printf("S(n) = 4 floor(log2 n) + 6 = %d\n", 4 * FloorLog2(n) + 6);

    // How many multiplications: about 2 log2 n instead of n (x = 1, to avoid overflow)
    printf("\n");
    printf("      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6\n");
    int values[] = {1, 2, 13, 100, 1000, 1000000};
    for (int v : values)
    {
        Reset();
        FastPower(1, v);
        printf("%7d | %8lld | %15lld | %4lld | %d\n", v, calls, mults, Total(), 4 * FloorLog2(v) + 6);
    }
    return 0;
}
#endif

Writing fast_power.cpp


In [2]:
!g++ -std=c++17 -O2 -o fast_power fast_power.cpp && ./fast_power

call                        | action
FastPower(3, 13)            | 13 != 0 => calls FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => calls FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => calls FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => calls FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | base case, returns 1
      FastPower(3, 1)       | y = 1, 1 is odd => returns 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 is odd => returns 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 is even => returns 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 is odd => returns 3 * 729 * 729 = 1594323
3^13 = 1594323, calls = 5, multiplications = 7
Line | Executions
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Total steps S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n |    calls | multiplications | S(n) | 4 floor(log2 n) + 6
      1 |        2 |   

**Try it yourself.** Change $x$ and $n$ and count the multiplications.

In [3]:
%%writefile prova_fast_power.cpp
#define SOLO_LIBRERIA   // do not run the main of fast_power.cpp
#include "fast_power.cpp"

int main()
{
    long long x = 2;
    int n = 10;        // <-- change x and n (with large x and n C++ and Java overflow)
    Reset();
    trace = true;
    printf("%-28s| %s\n", "call", "action");
    long long r = FastPower(x, n);
    trace = false;
    printf("%lld^%d = %lld, multiplications = %lld\n", x, n, r, mults);
    return 0;
}

Writing prova_fast_power.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_fast_power prova_fast_power.cpp && ./prova_fast_power

call                        | action
FastPower(2, 10)            | 10 != 0 => calls FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => calls FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => calls FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => calls FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | base case, returns 1
      FastPower(2, 1)       | y = 1, 1 is odd => returns 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 is even => returns 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 is odd => returns 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 is even => returns 32 * 32 = 1024
2^10 = 1024, multiplications = 6


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).